In [23]:
import time
import ctypes as ct
from ctypes import byref
import os
import sys
import matplotlib.pyplot as plt
from datetime import datetime


def save_hh_data(data, binning, offset, tacq, base_directory='D:/Data'):
    year = datetime.now().strftime('%Y')
    year_month = datetime.now().strftime('%Y%m')
    year_month_day = datetime.now().strftime('%Y%m%d')
    folder_path = os.path.join(base_directory, year, year_month, year_month_day, "g2")
    os.makedirs(folder_path, exist_ok=True)
    timestamp = datetime.now().strftime('%H%M%S')  # Using HHMMSS format for timestamp
    filename = f"hh_data_{timestamp}_bin{binning}_offset{offset}_tacq{tacq}.txt"
    file_path = os.path.join(folder_path, filename)
    with open(file_path, mode='w') as file:
        file.write("Index\tCounts\n")  # Header row
        for index, count in enumerate(data):
            file.write(f"{index}\t{count}\n")
    print(f"Data saved to {file_path}")
    return file_path


    


def closeDevices():
    MAXDEVNUM = 8
    if os.name == "nt":
        hhlib = ct.WinDLL("hhlib.dll")
    else:
        hhlib = ct.CDLL("libhh400.so")
    for i in range(0, MAXDEVNUM):
        hhlib.HH_CloseDevice(ct.c_int(i))
    sys.exit(0)

def get_hh_data(binning,offset,tacq):
    LIB_VERSION   = "3.0"
    MAXDEVNUM     = 8
    MODE_HIST     = 0
    MAXLENCODE    = 6
    HHMAXINPCHAN  = 8
    MAXHISTLEN    = 65536
    FLAG_OVERFLOW = 0x001
     
    syncDivider        = 1 # You can change this 
    syncChannelOffset  = 0 # You can change this (in ps, like a cable delay)
    inputChannelOffset = 0 # You can change this (in ps, like a cable delay)
    syncCFDZeroCross   = 10 # You can change this (in mV)
    syncCFDLevel       = 50 # You can change this (in mV)
    inputCFDZeroCross  = 10 # You can change this (in mV)
    inputCFDLevel      = 50 # You can change this (in mV)

    counts       = [(ct.c_uint * MAXHISTLEN)() for i in range(0, HHMAXINPCHAN)]
    dev          = []
    libVersion   = ct.create_string_buffer(b"", 8)
    hwSerial     = ct.create_string_buffer(b"", 8)
    hwPartno     = ct.create_string_buffer(b"", 8)
    hwVersion    = ct.create_string_buffer(b"", 8)
    hwModel      = ct.create_string_buffer(b"", 16)
    errorString  = ct.create_string_buffer(b"", 40)
    numChannels  = ct.c_int()
    histLen      = ct.c_int()
    resolution   = ct.c_double()
    syncRate     = ct.c_int()
    countRate    = ct.c_int()
    flags        = ct.c_int()
    warnings     = ct.c_int()
    warningstext = ct.create_string_buffer(b"", 16384)
    
    if os.name == "nt":
        hhlib = ct.WinDLL("hhlib.dll")
    else:
        hhlib = ct.CDLL("libhh400.so")

    def tryfunc(retcode, funcName, measRunning=False):
        if retcode < 0:
            hhlib.HH_GetErrorString(errorString, ct.c_int(retcode))
            print("HH_%s error %d (%s). Aborted." % (funcName, retcode,\
                  errorString.value.decode("utf-8")))
            closeDevices()
    
    hhlib.HH_GetLibraryVersion(libVersion)

    outputfile = open("histomodeout.txt", "w+")

    outputfile.write("Binning           : %d\n" % binning)
    outputfile.write("Offset            : %d\n" % offset)
    outputfile.write("AcquisitionTime   : %d\n" % tacq)
    outputfile.write("SyncDivider       : %d\n" % syncDivider)
    outputfile.write("SyncCFDZeroCross  : %d\n" % syncCFDZeroCross)
    outputfile.write("SyncCFDLevel      : %d\n" % syncCFDLevel)
    outputfile.write("InputCFDZeroCross : %d\n" % inputCFDZeroCross)
    outputfile.write("InputCFDLevel     : %d\n" % inputCFDLevel)

    for i in range(0, MAXDEVNUM):
        retcode = hhlib.HH_OpenDevice(ct.c_int(i), hwSerial)
        if retcode == 0:

            dev.append(i)
        # else:
        #     if retcode == -1: # HH_ERROR_DEVICE_OPEN_FAIL
        #         #print("  %1d        no device" % i)
        #     else:
        #         hhlib.HH_GetErrorString(errorString, ct.c_int(retcode))
        #         print("  %1d        %s" % (i, errorString.value.decode("utf8")))
        #
    if len(dev) < 1:
        print("No device available.")
        closeDevices()

    tryfunc(hhlib.HH_Initialize(ct.c_int(dev[0]), ct.c_int(MODE_HIST), ct.c_int(0)),\
            "Initialize")

    tryfunc(hhlib.HH_GetHardwareInfo(dev[0], hwModel, hwPartno, hwVersion),\
            "GetHardwareInfo")

    tryfunc(hhlib.HH_GetNumOfInputChannels(ct.c_int(dev[0]), byref(numChannels)),\
            "GetNumOfInputChannels")

    tryfunc(hhlib.HH_Calibrate(ct.c_int(dev[0])), "Calibrate")
    tryfunc(hhlib.HH_SetSyncDiv(ct.c_int(dev[0]), ct.c_int(syncDivider)), "SetSyncDiv")
    
    tryfunc(hhlib.HH_SetSyncCFD(ct.c_int(dev[0]), ct.c_int(syncCFDLevel),\
                                ct.c_int(syncCFDZeroCross)),\
            "SetSyncCFD")
    
    tryfunc(hhlib.HH_SetSyncChannelOffset(ct.c_int(dev[0]), ct.c_int(syncChannelOffset)),\
            "SetSyncChannelOffset")

    for i in range(0, numChannels.value):
        tryfunc(hhlib.HH_SetInputCFD(ct.c_int(dev[0]), ct.c_int(i), ct.c_int(inputCFDLevel),\
                                     ct.c_int(inputCFDZeroCross)),\
                "SetInputCFD")
    
        tryfunc(hhlib.HH_SetInputChannelOffset(ct.c_int(dev[0]), ct.c_int(i),\
                                           ct.c_int(inputChannelOffset)),\
            "SetInputChannelOffset")
    
    tryfunc(hhlib.HH_SetHistoLen(ct.c_int(dev[0]), ct.c_int(MAXLENCODE), byref(histLen)),\
            "SetHistoLen")

    
    tryfunc(hhlib.HH_SetBinning(ct.c_int(dev[0]), ct.c_int(binning)), "SetBinning")
    tryfunc(hhlib.HH_SetOffset(ct.c_int(dev[0]), ct.c_int(offset)), "SetOffset")
    tryfunc(hhlib.HH_GetResolution(ct.c_int(dev[0]), byref(resolution)), "GetResolution")
    print("Resolution is %1.1lfps" % resolution.value)

    time.sleep(0.4)
    
    tryfunc(hhlib.HH_GetSyncRate(ct.c_int(dev[0]), byref(syncRate)), "GetSyncRate")
    #print("\nSyncrate=%1d/s" % syncRate.value)
    
    for i in range(0, numChannels.value):
        tryfunc(hhlib.HH_GetCountRate(ct.c_int(dev[0]), ct.c_int(i), byref(countRate)),\
                "GetCountRate")
        print("Countrate[%1d]=%1d/s" % (i, countRate.value))
    
    # New from v1.2: after getting the count rates you can check for warnings
    tryfunc(hhlib.HH_GetWarnings(ct.c_int(dev[0]), byref(warnings)), "GetWarnings")
    if warnings.value != 0:
        hhlib.HH_GetWarningsText(ct.c_int(dev[0]), warningstext, warnings)
        #print("\n\n%s" % warningstext.value.decode("utf-8"))

    tryfunc(hhlib.HH_SetStopOverflow(ct.c_int(dev[0]), ct.c_int(0), ct.c_int(10000)),\
            "SetStopOverflow") # for example only

    tryfunc(hhlib.HH_GetSyncRate(ct.c_int(dev[0]), byref(syncRate)), "GetSyncRate")
    #print("Syncrate=%1d/s" % syncRate.value)

    for i in range(0, numChannels.value):
        tryfunc(hhlib.HH_GetCountRate(ct.c_int(dev[0]), ct.c_int(i), byref(countRate)),\
                "GetCountRate")
        #print("Countrate[%1d]=%1d/s" % (i, countRate.value))

    tryfunc(hhlib.HH_StartMeas(ct.c_int(dev[0]), ct.c_int(tacq)), "StartMeas") #MEAS STARTS HERE !!!!!!!!
    print("\nMeasuring for %1d milliseconds..." % tacq)

    ctcstatus = ct.c_int(0)
    while ctcstatus.value == 0:
        tryfunc(hhlib.HH_CTCStatus(ct.c_int(dev[0]), byref(ctcstatus)),\
                "CTCStatus")

    tryfunc(hhlib.HH_StopMeas(ct.c_int(dev[0])), "StopMeas")

    for i in range(0, numChannels.value):
        tryfunc(hhlib.HH_GetHistogram(ct.c_int(dev[0]), byref(counts[i]),\
                                  ct.c_int(i), ct.c_int(0)),\
            "GetHistogram")

        integralCount = 0
        for j in range(0, histLen.value):
            integralCount += counts[i][j]

        #print("  Integralcount[%1d]=%1.0lf" % (i,integralCount))

    tryfunc(hhlib.HH_GetFlags(ct.c_int(dev[0]), byref(flags)), "GetFlags")

    if flags.value & FLAG_OVERFLOW > 0:
        print("  Overflow.")


    plt.plot(counts[0])
    plt.show()
    return(counts[0])



binning= 7 # You can change this
offset = 0
tacq = 3000 # Measurement time in millisec, you can change this


data = get_hh_data(binning, offset, tacq)
save_hh_data(data, binning, offset, tacq)

print(len(data))
closeDevices()



Resolution is 128.0ps
Countrate[0]=7960/s
Countrate[1]=0/s
Countrate[2]=0/s
Countrate[3]=0/s

Measuring for 3000 milliseconds...


SystemExit: 0

In [21]:
print(data[25000])

0
